# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Published Date: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the available record sets and their fields using their `@id` values for referencing. We print a sample record from each record set.

In [ ]:
# The dataset provides access through record sets.
record_sets = dataset.record_sets
print("Available Record Sets and Fields:")
record_sets_ids = []

for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")
    record_sets_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, type: {field.data_type}")
    # Print one sample record
    records = list(dataset.records(record_set=record_set.id))
    if records:
        print("  Sample record:")
        print(records[0])
    else:
        print("  No records found.")
    print("---")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. All references use the `@id` field, which uniquely identifies entities in the dataset.

In [ ]:
# Extract data from each record set, store in a dictionary
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head(2), "\n")
    else:
        print(f"No records found for RecordSet {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, let's select a numeric field from the main record set, filter by a threshold, normalize values, and group by a demographic field (e.g., `gender`). All fields referenced by their `@id`.

In [ ]:
# For this example, use the first RecordSet (typically the main survey table)
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    df = dataframes[main_record_set_id]

    # Identify candidate numeric fields by checking types
    numeric_fields = []
    for field in dataset.record_set(main_record_set_id).fields:
        if field.data_type == 'schema:Integer' or field.data_type == 'schema:Float' or field.data_type == 'schema:Number':
            numeric_fields.append(field.id)

    print("Numeric fields:", numeric_fields)
    # Pick first available numeric field (e.g., GAD-7 total score): replace with actual @id as appropriate
    numeric_field_id = numeric_fields[0] if numeric_fields else None
    # Choose a grouping field (demographic, e.g., gender): reference by @id
    group_field_id = None
    for field in dataset.record_set(main_record_set_id).fields:
        if 'gender' in field.name.lower():
            group_field_id = field.id

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by the demographic field if it exists
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("Suitable numeric field not found for EDA.")
else:
    print("No record set found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

We'll plot the distribution of a numeric field (e.g., GAD-7 score) and a barplot for its mean grouped by gender (using the `@id` values).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if record_sets_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Barplot: mean by group_field
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        plt.figure(figsize=(6,4))
        sns.barplot(x=group_means.index.astype(str), y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Cannot visualize: missing numeric field.")

## 6. Conclusion
Summarize key findings from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides rich information on demographic and psychological factors for residents of Kilifi County.
- Using `mlcroissant`, we were able to programmatically extract record sets, reference data fields and columns by their `@id`, and analyze distributions and group statistics.
- The approach demonstrated lets users flexibly work with FAIR data packages in reproducible workflows.

Feel free to extend this notebook for deeper analyses, modeling, or integration with other croissant datasets!